In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# ═══════════════════════════════════════════════════════
# CELL 1: INSTALLS
# ═══════════════════════════════════════════════════════
!pip install xgboost scikit-learn pandas numpy matplotlib seaborn tqdm --quiet
print("✅ All packages installed.")

✅ All packages installed.


In [3]:
# ═══════════════════════════════════════════════════════
# CELL 2: IMPORTS & CONFIG
# ═══════════════════════════════════════════════════════
import os, gc, time, warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from tqdm.notebook import tqdm

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report
from sklearn.feature_selection import RFE, RFECV

import xgboost as xgb

warnings.filterwarnings('ignore')
np.random.seed(42)

# ── PATH TO YOUR DATASET ──────────────────────────────────────────────────────
DATA_PATH = '/content/drive/MyDrive/final_dataset/iot_cleaned_sampled_smote(hybrid40-40)/preprocessed_dataset_with_shadow.csv'

# ── OUTPUT FOLDER FOR FINAL ANALYSIS ─────────────────────────────────────────
FS_OUTPUT_FOLDER = '/content/drive/MyDrive/final_dataset/fs'

# ── SUPERCLASS MAPPING (multi-label → 8 superclasses) ────────────────────────
SUPERCLASS_MAP = {
    'DDOS-ICMP_FLOOD': 'DDoS', 'DDOS-UDP_FLOOD': 'DDoS', 'DDOS-TCP_FLOOD': 'DDoS',
    'DDOS-PSHACK_FLOOD': 'DDoS', 'DDOS-SYN_FLOOD': 'DDoS', 'DDOS-RSTFINFLOOD': 'DDoS',
    'DDOS-SYNONYMOUSIP_FLOOD': 'DDoS', 'DDOS-UDP_FRAGMENTATION': 'DDoS', 'DDOS-ACK_FRAGMENTATION': 'DDoS',
    'DDOS-ICMP_FRAGMENTATION': 'DDoS', 'DDOS-HTTP_FLOOD': 'DDoS',
    'DOS-UDP_FLOOD': 'DoS', 'DOS-TCP_FLOOD': 'DoS', 'DOS-SYN_FLOOD': 'DoS', 'DOS-HTTP_FLOOD': 'DoS',
    'MIRAI-GREETH_FLOOD': 'Mirai', 'MIRAI-UDPPLAIN': 'Mirai', 'MIRAI-GREIP_FLOOD': 'Mirai',
    'RECON-HOSTDISCOVERY': 'Recon', 'RECON-OSSCAN': 'Recon', 'RECON-PORTSCAN': 'Recon', 'RECON-PINGSWEEP': 'Recon',
    'VULNERABILITYSCAN': 'Recon', 'SQLINJECTION': 'Web-based', 'BACKDOOR_MALWARE': 'Web-based',
    'XSS': 'Web-based', 'BROWSERHIJACKING': 'Web-based', 'COMMANDINJECTION': 'Web-based',
    'UPLOADING_ATTACK': 'Web-based', 'DNS_SPOOFING': 'Spoofing', 'MITM-ARPSPOOFING': 'Spoofing',
    'DICTIONARYBRUTEFORCE': 'Brute Force', 'BENIGN': 'Benign'
}

# ── BASELINE RESULTS (from your XGBoost 15-feature run) ──────────────────────
BASELINE_RESULTS = {
    'Benign':      {'samples': 205326, 'detection_rate': 95.82, 'missed': 8584},
    'DDoS':        {'samples': 88000,  'detection_rate': 100.00, 'missed': 1},
    'Web-based':   {'samples': 48000,  'detection_rate': 93.84, 'missed': 2958},
    'Recon':       {'samples': 40000,  'detection_rate': 62.05, 'missed': 15181},
    'DoS':         {'samples': 32000,  'detection_rate': 100.00, 'missed': 0},
    'Mirai':       {'samples': 24000,  'detection_rate': 100.00, 'missed': 0},
    'Spoofing':    {'samples': 16000,  'detection_rate': 73.77, 'missed': 4197},
    'Brute Force': {'samples': 8000,   'detection_rate': 79.76, 'missed': 1619},
}
BASELINE_ACCURACY = 93.0662

print("✅ Config loaded.")
print(f"   Baseline XGBoost accuracy : {BASELINE_ACCURACY}%")
print(f"   Superclasses tracked      : {list(set(SUPERCLASS_MAP.values()))}")

✅ Config loaded.
   Baseline XGBoost accuracy : 93.0662%
   Superclasses tracked      : ['Benign', 'Mirai', 'Spoofing', 'Brute Force', 'DoS', 'DDoS', 'Recon', 'Web-based']


In [4]:
# ═══════════════════════════════════════════════════════
# CELL 3: LOAD & PREPARE DATA
# ═══════════════════════════════════════════════════════


print(f"Loading dataset from:\n  {DATA_PATH}\n")
df = pd.read_csv(DATA_PATH)
print(f"  Shape (before clean) : {df.shape}")

# ── Drop rows with missing labels ─────────────────────────────────────────────
df = df.dropna(subset=['Superclass', 'Binary_Label'])
print(f"  Shape (after clean)  : {df.shape}")
print(f"  Columns              : {list(df.columns)}")

# ── Feature columns (drop all non-feature cols) ───────────────────────────────
DROP_COLS = ['Fine_Label', 'Binary_Label', 'Superclass']
feature_cols = [c for c in df.columns if c not in DROP_COLS]
print(f"\n  Feature columns ({len(feature_cols)}): {feature_cols}")

# ── Use pre-existing label columns — no remapping needed ─────────────────────
df['binary_label'] = df['Binary_Label'].astype(int)
df['superclass']   = df['Superclass'].astype(str)

X       = df[feature_cols].copy()
y       = df['binary_label'].copy()
y_super = df['superclass'].copy()

print(f"\n  Binary label distribution:")
print(f"    Benign (0) : {(y==0).sum():,}")
print(f"    Attack (1) : {(y==1).sum():,}")
print(f"\n  Superclass distribution:")
print(df['Superclass'].value_counts())

Loading dataset from:
  /content/drive/MyDrive/final_dataset/iot_cleaned_sampled_smote(hybrid40-40)/preprocessed_dataset_with_shadow.csv

  Shape (before clean) : (2346628, 18)
  Shape (after clean)  : (2306628, 18)
  Columns              : ['number', 'https', 'ack_flag_number', 'time_to_live', 'ack_count', 'psh_flag_number', 'header_length', 'iat', 'tot sum', 'rate', 'max', 'avg', 'syn_flag_number', 'variance', 'udp', 'Fine_Label', 'Binary_Label', 'Superclass']

  Feature columns (15): ['number', 'https', 'ack_flag_number', 'time_to_live', 'ack_count', 'psh_flag_number', 'header_length', 'iat', 'tot sum', 'rate', 'max', 'avg', 'syn_flag_number', 'variance', 'udp']

  Binary label distribution:
    Benign (0) : 1,026,628
    Attack (1) : 1,280,000

  Superclass distribution:
Superclass
Benign         1026628
DDoS            440000
Web-based       240000
Recon           200000
DoS             160000
Mirai           120000
Spoofing         80000
Brute Force      40000
Name: count, dtype:

In [5]:
# ═══════════════════════════════════════════════════════
# CELL 4: TRAIN / TEST SPLIT  (80 / 20, stratified)
# ═══════════════════════════════════════════════════════
X_train, X_test, y_train, y_test, idx_train, idx_test = train_test_split(
    X, y, df.index, test_size=0.2, random_state=42, stratify=y
)

# Also grab superclass labels for the test set
y_super_test = y_super.loc[idx_test]

print(f"Train size : {len(X_train):,}")
print(f"Test  size : {len(X_test):,}")
print(f"\nFeatures available (15) : {list(X.columns)}")

Train size : 1,845,302
Test  size : 461,326

Features available (15) : ['number', 'https', 'ack_flag_number', 'time_to_live', 'ack_count', 'psh_flag_number', 'header_length', 'iat', 'tot sum', 'rate', 'max', 'avg', 'syn_flag_number', 'variance', 'udp']


In [6]:
# ═══════════════════════════════════════════════════════
# CELL 5: SHARED EVALUATION HELPER
#   Trains XGBoost on given feature subset, returns full
#   metrics dict including per-superclass detection rates
# ═══════════════════════════════════════════════════════

XGB_PARAMS = dict(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric='logloss',
    random_state=42,
    tree_method='hist',
    n_jobs=-1
)

def evaluate_features(feature_subset, label=""):
    start = time.time()
    model = xgb.XGBClassifier(**XGB_PARAMS)
    model.fit(X_train[feature_subset], y_train, verbose=False)
    elapsed = time.time() - start

    preds = model.predict(X_test[feature_subset])
    overall_acc = accuracy_score(y_test, preds) * 100

    # ── Per-superclass detection rate ────────────────────────────────────────
    test_df = pd.DataFrame({
        'superclass': y_super_test.values,
        'y_true':     y_test.values,
        'y_pred':     preds
    }).dropna(subset=['superclass'])

    test_df['superclass'] = test_df['superclass'].astype(str)

    results = {}
    for sc in sorted(test_df['superclass'].unique()):
        sc_df = test_df[test_df['superclass'] == sc]
        n = len(sc_df)
        if sc.lower() == 'benign':
            correct  = (sc_df['y_pred'] == sc_df['y_true']).sum()
            det_rate = correct / n * 100
            missed   = n - correct
        else:
            detected     = ((sc_df['y_true'] == 1) & (sc_df['y_pred'] == 1)).sum()
            total_attack = (sc_df['y_true'] == 1).sum()
            det_rate     = detected / total_attack * 100 if total_attack > 0 else 0
            missed       = total_attack - detected
        results[sc] = {
            'samples':        n,
            'detection_rate': round(det_rate, 2),
            'missed':         int(missed)
        }

    return {
        'label':          label,
        'features':       feature_subset,
        'n_features':     len(feature_subset),
        'accuracy':       round(overall_acc, 4),
        'inference_time': round(elapsed, 4),
        'throughput':     round(len(X_test) / elapsed),
        'per_class':      results,
        'model':          model
    }

print("✅ evaluate_features() ready.")
print("   XGB params:", XGB_PARAMS)

✅ evaluate_features() ready.
   XGB params: {'n_estimators': 200, 'max_depth': 6, 'learning_rate': 0.1, 'subsample': 0.8, 'colsample_bytree': 0.8, 'eval_metric': 'logloss', 'random_state': 42, 'tree_method': 'hist', 'n_jobs': -1}


In [7]:
# ═══════════════════════════════════════════════════════
# CELL 6: BASELINE — ALL 15 FEATURES  (replication)
# ═══════════════════════════════════════════════════════
ALL_FEATURES = list(X.columns)
print(f"Features ({len(ALL_FEATURES)}): {ALL_FEATURES}\n")

print("Training XGBoost with ALL 15 features (baseline replication)...")
baseline_result = evaluate_features(ALL_FEATURES, label="Baseline (All 15 Features)")

SUPERCLASSES = sorted(baseline_result['per_class'].keys())
print(f"\nSuperclasses detected: {SUPERCLASSES}")

print("\n" + "="*65)
print(f"  {'BASELINE — ALL 15 FEATURES':^61}")
print("="*65)
print(f"  Overall Accuracy     : {baseline_result['accuracy']}%")
print(f"  Inference Time       : {baseline_result['inference_time']}s")
print(f"  Throughput           : {baseline_result['throughput']:,} samples/sec")
print("="*65)
print(f"  {'Superclass':<15} {'Samples':>10} {'Detection %':>12} {'Missed':>8}")
print("-"*65)
for sc, m in baseline_result['per_class'].items():
    print(f"  {sc:<15} {m['samples']:>10,} {m['detection_rate']:>11.2f}% {m['missed']:>8,}")
print("="*65)

Features (15): ['number', 'https', 'ack_flag_number', 'time_to_live', 'ack_count', 'psh_flag_number', 'header_length', 'iat', 'tot sum', 'rate', 'max', 'avg', 'syn_flag_number', 'variance', 'udp']

Training XGBoost with ALL 15 features (baseline replication)...

Superclasses detected: ['Benign', 'Brute Force', 'DDoS', 'DoS', 'Mirai', 'Recon', 'Spoofing', 'Web-based']

                   BASELINE — ALL 15 FEATURES                  
  Overall Accuracy     : 93.0752%
  Inference Time       : 52.6704s
  Throughput           : 8,759 samples/sec
  Superclass         Samples  Detection %   Missed
-----------------------------------------------------------------
  Benign             205,326       96.06%    8,080
  Brute Force          8,124       80.51%    1,583
  DDoS                87,976      100.00%        2
  DoS                 31,778      100.00%        0
  Mirai               24,110      100.00%        0
  Recon               40,058       62.08%   15,189
  Spoofing            15,898   

In [8]:
# ═══════════════════════════════════════════════════════
# CELL 7: RECURSIVE FEATURE ELIMINATION (RFE)
#   Uses sklearn RFE with XGBoost estimator.
#   We sweep k = 1..15 to find best k, then do full eval.
# ═══════════════════════════════════════════════════════
from sklearn.feature_selection import RFE

print("Starting RFE sweep over k = 1 to 15 features...\n")

rfe_base_model = xgb.XGBClassifier(**XGB_PARAMS)
rfe_sweep = []

for k in tqdm(range(1, len(ALL_FEATURES)+1), desc="RFE k sweep"):
    rfe = RFE(estimator=xgb.XGBClassifier(**XGB_PARAMS), n_features_to_select=k, step=1)
    rfe.fit(X_train, y_train)
    selected = [f for f, s in zip(ALL_FEATURES, rfe.support_) if s]
    m = evaluate_features(selected)
    rfe_sweep.append((k, m['accuracy'], selected))
    print(f"  k={k:2d}: {selected} → {m['accuracy']:.4f}%")

# ── Find best k ───────────────────────────────────────────────────────────────
best_rfe = max(rfe_sweep, key=lambda x: x[1])
rfe_opt_k      = best_rfe[0]
rfe_opt_acc    = best_rfe[1]
rfe_opt_feats  = best_rfe[2]

# Get RFE ranking for all features (using best k's estimator)
rfe_final = RFE(estimator=xgb.XGBClassifier(**XGB_PARAMS),
                n_features_to_select=rfe_opt_k, step=1)
rfe_final.fit(X_train, y_train)
rfe_ranking = dict(zip(ALL_FEATURES, rfe_final.ranking_))

print("\n" + "="*65)
print(f"  RFE OPTIMAL RESULT: {rfe_opt_k} features → {rfe_opt_acc:.4f}% accuracy")
print(f"  Optimal feature set: {rfe_opt_feats}")
print(f"  Feature rankings (1=selected):")
for feat, rank in sorted(rfe_ranking.items(), key=lambda x: x[1]):
    print(f"    Rank {rank:2d}: {feat}")
print("="*65)

print("\nRunning full evaluation on RFE-selected features...")
rfe_result = evaluate_features(rfe_opt_feats, label=f"RFE ({rfe_opt_k} features)")

Starting RFE sweep over k = 1 to 15 features...



RFE k sweep:   0%|          | 0/15 [00:00<?, ?it/s]

  k= 1: ['number'] → 75.5576%
  k= 2: ['number', 'https'] → 86.3483%
  k= 3: ['number', 'https', 'psh_flag_number'] → 88.3724%
  k= 4: ['number', 'https', 'psh_flag_number', 'syn_flag_number'] → 89.2473%
  k= 5: ['number', 'https', 'psh_flag_number', 'max', 'syn_flag_number'] → 91.5923%
  k= 6: ['number', 'https', 'ack_flag_number', 'psh_flag_number', 'max', 'syn_flag_number'] → 91.8333%
  k= 7: ['number', 'https', 'ack_flag_number', 'ack_count', 'psh_flag_number', 'max', 'syn_flag_number'] → 91.8474%
  k= 8: ['number', 'https', 'ack_flag_number', 'ack_count', 'psh_flag_number', 'max', 'syn_flag_number', 'udp'] → 92.0900%
  k= 9: ['number', 'https', 'ack_flag_number', 'ack_count', 'psh_flag_number', 'iat', 'max', 'syn_flag_number', 'udp'] → 92.6163%
  k=10: ['number', 'https', 'ack_flag_number', 'time_to_live', 'ack_count', 'psh_flag_number', 'iat', 'max', 'syn_flag_number', 'udp'] → 92.8638%
  k=11: ['number', 'https', 'ack_flag_number', 'time_to_live', 'ack_count', 'psh_flag_number',

In [9]:
# ═══════════════════════════════════════════════════════
# CELL 8: SAVE RFE RESULTS FOR FINAL ANALYSIS
# ═══════════════════════════════════════════════════════
import pickle, os

os.makedirs(FS_OUTPUT_FOLDER, exist_ok=True)

rfe_payload = {
    'rfe_result':     rfe_result,
    'rfe_sweep':      rfe_sweep,
    'rfe_opt_k':      rfe_opt_k,
    'rfe_opt_acc':    rfe_opt_acc,
    'rfe_opt_feats':  rfe_opt_feats,
    'rfe_ranking':    rfe_ranking,
    'baseline_result': baseline_result,
    'ALL_FEATURES':   ALL_FEATURES,
}

save_path = os.path.join(FS_OUTPUT_FOLDER, 'rfe_results.pkl')
with open(save_path, 'wb') as f:
    pickle.dump(rfe_payload, f)

print(f"✅ RFE results saved to: {save_path}")
print(f"   Optimal k          : {rfe_opt_k}")
print(f"   Optimal accuracy   : {rfe_opt_acc:.4f}%")
print(f"   Optimal features   : {rfe_opt_feats}")

✅ RFE results saved to: /content/drive/MyDrive/final_dataset/fs/rfe_results.pkl
   Optimal k          : 14
   Optimal accuracy   : 93.0923%
   Optimal features   : ['number', 'https', 'ack_flag_number', 'time_to_live', 'ack_count', 'psh_flag_number', 'header_length', 'iat', 'tot sum', 'rate', 'max', 'syn_flag_number', 'variance', 'udp']


In [11]:
# ═══════════════════════════════════════════════════════════════════════
# CELL: DETAILED FINAL BREAKDOWN  —  Baseline vs RFE Optimal
#   Compares XGBoost (all 15 features, 93% reference) against the
#   best feature subset found by RFE.
#   Shows: per-superclass detection rates + deltas, F1, FNR, FPR.
#   Also augments and re-saves the pkl with these extra metrics.
# ═══════════════════════════════════════════════════════════════════════
from sklearn.metrics import f1_score, recall_score, confusion_matrix
import pickle, os

def detailed_breakdown(result, title, compare=None):
    pc         = result["per_class"]
    feats      = result["features"]
    n_feats    = result["n_features"]
    acc        = result["accuracy"]
    inf_time   = result["inference_time"]
    throughput = result["throughput"]

    W = 76
    sep = "=" * W
    print()
    print(sep)
    print("  " + title)
    print("  Features ({}): {}".format(n_feats, feats))
    print(sep)

    has_delta = compare is not None
    hdr = "  {:<16} {:>8} {:>13} {:>8}".format(
        "Superclass", "Samples", "Detection %", "Missed")
    if has_delta:
        hdr += "  {:>15}".format("Delta vs Base")
    print(hdr)
    print("  " + "-" * (72 if not has_delta else 90))

    for sc, m in sorted(pc.items(), key=lambda x: -x[1]["samples"]):
        dr = m["detection_rate"]
        delta_str = ""
        if has_delta and sc in compare["per_class"]:
            delta = dr - compare["per_class"][sc]["detection_rate"]
            arrow = "+" if delta > 0 else ("-" if delta < 0 else " ")
            delta_str = "  {} {:.2f}%".format(arrow, delta)
        print("  {:<16} {:>8,} {:>12.2f}% {:>8}{}".format(
            sc, m["samples"], dr, m["missed"], delta_str))

    print("  " + "-" * 72)

    if "model" in result and result["model"] is not None:
        preds = result["model"].predict(X_test[feats])
        f1_v  = f1_score(y_test, preds)
        rec_v = recall_score(y_test, preds)
        tn, fp, fn, tp = confusion_matrix(y_test, preds).ravel()
        fnr_v = fn / (fn + tp) * 100 if (fn + tp) > 0 else 0
        fpr_v = fp / (fp + tn) * 100 if (fp + tn) > 0 else 0
    else:
        f1_v = rec_v = fnr_v = fpr_v = float("nan")

    result["f1"]  = round(f1_v,  4)
    result["rec"] = round(rec_v, 4)
    result["fnr"] = round(fnr_v, 3)
    result["fpr"] = round(fpr_v, 3)

    acc_delta_str = ""
    if has_delta:
        d = acc - compare["accuracy"]
        acc_delta_str = "  (Delta {:.4f}%)".format(d)

    print()
    print("  Overall Accuracy  : {:.4f}%{}".format(acc, acc_delta_str))
    print("  F1-Score          : {:.4f}".format(f1_v))
    print("  Recall (DR)       : {:.4f}".format(rec_v))
    print("  FNR               : {:.3f}%".format(fnr_v))
    print("  FPR               : {:.3f}%".format(fpr_v))
    print("  Inference Time    : {}s".format(inf_time))
    print("  Throughput        : {:,} samples/s".format(throughput))
    print(sep)
    return result

# ── Baseline ────────────────────────────────────────────────────────────────
detailed_breakdown(baseline_result, "BASELINE  —  XGBoost, All 15 Features (93% Reference Model)")

# ── RFE Optimal ─────────────────────────────────────────────────────────────
rfe_result = detailed_breakdown(
    rfe_result,
    "RFE  —  Optimal {} Features  vs  Baseline".format(rfe_opt_k),
    compare=baseline_result
)

# ── Re-save pkl with augmented metrics ──────────────────────────────────────
rfe_payload_v2 = {
    "rfe_result":     rfe_result,
    "rfe_sweep":      rfe_sweep,
    "rfe_opt_k":      rfe_opt_k,
    "rfe_opt_acc":    rfe_opt_acc,
    "rfe_opt_feats":  rfe_opt_feats,
    "rfe_ranking":    rfe_ranking,
    "baseline_result": baseline_result,
    "ALL_FEATURES":   ALL_FEATURES,
}
save_path = os.path.join(FS_OUTPUT_FOLDER, "rfe_results.pkl")
with open(save_path, "wb") as fh:
    pickle.dump(rfe_payload_v2, fh)
print("Updated RFE pkl re-saved to: " + save_path)


  BASELINE  —  XGBoost, All 15 Features (93% Reference Model)
  Features (15): ['number', 'https', 'ack_flag_number', 'time_to_live', 'ack_count', 'psh_flag_number', 'header_length', 'iat', 'tot sum', 'rate', 'max', 'avg', 'syn_flag_number', 'variance', 'udp']
  Superclass        Samples   Detection %   Missed
  ------------------------------------------------------------------------
  Benign            205,326        96.06%     8080
  DDoS               87,976       100.00%        2
  Web-based          48,056        94.03%     2871
  Recon              40,058        62.08%    15189
  DoS                31,778       100.00%        0
  Mirai              24,110       100.00%        0
  Spoofing           15,898        73.45%     4221
  Brute Force         8,124        80.51%     1583
  ------------------------------------------------------------------------

  Overall Accuracy  : 93.0752%
  F1-Score          : 0.9356
  Recall (DR)       : 0.9068
  FNR               : 9.323%
  FPR     